# 07. EDA — AI 노출도와 청년 고용 (E1~E9)

설계 문서 `docs/ai_exposure_design_eda.md` 4절을 실행한다. 원본은 `data/raw/`(2026-11-11 수집)에 있고, 경로는 `src/config.py`를 따른다. 그림은 `reports/figures/`에 저장한다.

- 결과 변수: KOSIS `DT_1ES3B28J` 직업 중분류 × 연령 취업자 (반기, 2013 상~2026 상)
- 처치 강도: ILO WP140 직업별 생성형 AI 노출도 → KSCO 8차 중분류
- 시간 변수: Epoch Capabilities Index (ECI)

E2(월별 공표치 재계산)는 월별 표를 받지 않아 이번 실행에서 제외했다.

In [ ]:
import sys

sys.path.append("..")  # 노트북 폴더 기준으로 src를 찾는다 (README 6절)
from src import config

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import font_manager
from matplotlib.patches import Patch

try:
    display
except NameError:  # 스크립트로 실행할 때
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

RAW, OUT, FIG = config.DATA_RAW, config.DATA_PROCESSED, config.FIGURES
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(exist_ok=True)

# 색: 글자는 잉크 색, 노출도 3분위는 한 색상의 밝기 단계(순서형)
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUE = "#2a78d6"
ORDER = ["하", "중", "상"]
TERC = {"하": "#86b6ef", "중": "#2a78d6", "상": "#0d366b"}

_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT = next(f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "DejaVu Sans"] if f in _fonts)
sns.set_theme(style="whitegrid", font=FONT, rc={
    "axes.unicode_minus": False, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.labelsize": 9,
    "axes.titleweight": "bold", "axes.titlesize": 10, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})
CHATGPT = 2022 + 11 / 12  # 2022-11-30 출시


def mark_chatgpt(ax, label=True):
    ax.axvline(CHATGPT, color=MUTED, lw=1, ls="--", zorder=1)
    if label:
        ax.annotate("ChatGPT 출시", (CHATGPT, 1), xycoords=("data", "axes fraction"), xytext=(-4, -3),
                    textcoords="offset points", ha="right", va="top", fontsize=7.5, color=INK2)


def end_label(ax, x, y, text):
    ax.annotate(text, (x, y), xytext=(5, 0), textcoords="offset points", va="center", fontsize=8, color=INK2)

## E1. 패널 구성과 구조 점검

KOSIS CSV는 머리글이 2행(시점, 항목)이고, 값이 전부 빈 항목 열은 시점마다 빠져 있다. 열 위치가 아니라 머리글로 읽는다.

In [ ]:
raw = pd.read_csv(RAW / "kosis" / "DT_1ES3B28J_직업중분류x연령_2013H1-2026H1.csv", encoding="cp949", header=None, dtype=str)
prd = raw.iloc[0, 1:].str.split().str[0].str[1:]   # 'H201301 2013.1/2' → '201301'
itm = raw.iloc[1, 1:].str.split().str[0]            # 'T20 연령(15~29세)' → 'T20'
long = raw.iloc[2:].melt(id_vars=[0], var_name="col", value_name="v").rename(columns={0: "label"})
long["code"] = long.label.str.split().str[0]
long["name"] = long.label.str.split(n=1).str[1].str.replace("?", "·", regex=False)  # CP949에서 깨진 가운뎃점
long["period"], long["item"] = long.col.map(prd), long.col.map(itm)

ITEMS = {"T00": "total", "T20": "y15_29", "T21": "a30_39", "T22": "a40_49", "T23": "a50_59", "T24": "a60p"}
cnt = long[long.item.isin(ITEMS)].assign(n=lambda d: pd.to_numeric(d.v, errors="coerce").fillna(0))  # '-'(해당 없음) → 0
panel = cnt.pivot(index=["code", "name", "period"], columns="item", values="n").rename(columns=ITEMS).reset_index()
panel.columns.name = None
flag = long.loc[long.item == "T201", ["code", "period", "v"]].assign(y_flag=lambda d: d.v.eq("*")).drop(columns="v")
panel = panel.merge(flag, on=["code", "period"], how="left")
panel["y_flag"] = panel.y_flag.fillna(False).astype(bool)   # 청년 추정치 RSE 25% 이상
panel["a30_59"] = panel[["a30_39", "a40_49", "a50_59"]].sum(axis=1)
panel["year"], panel["half"] = panel.period.str[:4].astype(int), panel.period.str[4:].astype(int)
panel["time"] = panel.year + np.where(panel.half == 1, 3.5, 9.5) / 12   # 조사 기준월: 4월·10월
panel["post"] = panel.period >= "202301"

total = panel[panel.code == "0"].set_index("period")
major = panel[(panel.code.str.len() == 1) & (panel.code != "0")]
mid = panel[panel.code.str.len() == 2].copy()
mid["major"] = mid.code.str[0]

d_mid = (mid.groupby(["major", "period"]).total.sum() - major.set_index(["code", "period"]).total).abs().max()
d_maj = (major.groupby("period").total.sum() - total.total).abs().max()
d_age = (panel[["y15_29", "a30_39", "a40_49", "a50_59", "a60p"]].sum(axis=1) - panel.total).abs().max()
print(f"중분류 {mid.code.nunique()}개 × 시점 {mid.period.nunique()}개 = {len(mid):,}셀")
print(f"합계 차이(천명): 중분류→대분류 {d_mid:.0f}, 대분류→계 {d_maj:.0f}, 연령 합→취업자 {d_age:.0f}")
assert max(d_mid, d_maj, d_age) <= 3, "합계가 반올림 범위를 벗어남"

## E3. 결과 변수 추이

In [ ]:
base22 = total[total.year == 2022][["y15_29", "a30_59", "a60p"]].mean()
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for col, lab, color, lw in [("y15_29", "15~29세", BLUE, 2.4), ("a30_59", "30~59세", INK2, 1.6), ("a60p", "60세 이상", MUTED, 1.6)]:
    y = 100 * total[col] / base22[col]
    ax.plot(total.time, y, color=color, lw=lw, label=lab)
    end_label(ax, total.time.iloc[-1], y.iloc[-1], lab)
ax.axhline(100, color=AXIS, lw=0.8, zorder=1)
mark_chatgpt(ax)
ax.set(title="연령대별 취업자 (2022년 평균 = 100)", xlabel="", ylabel="지수", xlim=(2013, 2027.6))
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.savefig(FIG / "fig1_연령대별_취업자.png")

display(total.loc[["201301", "201902", "202202", "202402", "202601"], ["total", "y15_29", "a30_59", "a60p"]].astype(int))

In [ ]:
g1 = sns.relplot(data=major.sort_values(["code", "time"]), x="time", y="y15_29", col="name", col_wrap=3, kind="line",
                 color=BLUE, height=1.9, aspect=1.55, facet_kws={"sharey": False})
g1.set_titles("{col_name}", size=8.5)
g1.set_axis_labels("", "천명")
g1.set(xticks=[2014, 2018, 2022, 2026])
for ax in g1.axes.flat:
    mark_chatgpt(ax, label=False)
g1.figure.suptitle("대분류별 청년(15~29세) 취업자 — 점선은 ChatGPT 출시", x=0.01, y=1.03, ha="left", fontsize=10, fontweight="bold")
g1.savefig(FIG / "fig2_대분류별_청년취업자.png")

## E4. 청년 셀 신뢰도

KOSIS는 상대표준오차(RSE)가 25% 이상인 추정치에 `*`를 붙인다.

In [ ]:
mid["label"] = mid.code + " " + mid.name
hm = mid.pivot(index="label", columns="period", values="y_flag").astype(int)
hm.columns = [f"{p[2:4]}{'상' if p[4:] == '01' else '하'}" for p in hm.columns]
fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(hm, cmap=["#f0efec", BLUE], cbar=False, linewidths=0.8, linecolor=SURFACE, ax=ax)
ax.set(title="청년 취업자 추정치의 신뢰도: 중분류 × 시점", xlabel="", ylabel="")
ax.tick_params(axis="y", labelsize=7, length=0)
ax.tick_params(axis="x", labelsize=7, length=0, rotation=90)
ax.legend(handles=[Patch(color=BLUE, label="RSE 25% 이상"), Patch(color="#f0efec", label="신뢰 가능")],
          frameon=False, fontsize=8, ncol=2, loc="lower right", bbox_to_anchor=(1, 1.0))
fig.savefig(FIG / "fig3_청년셀_신뢰도.png")

share_cells = mid.y_flag.mean()
share_emp = mid.loc[mid.y_flag, "y15_29"].sum() / mid.y15_29.sum()
often = mid.groupby("code").y_flag.mean().loc[lambda s: s >= 0.5].index.tolist()
print(f"표시된 셀 {mid.y_flag.sum()}/{len(mid)} ({share_cells:.1%}), 그 셀의 청년 취업자 비중 {share_emp:.2%}")
print("시점의 절반 이상에서 표시된 중분류:", often)

## E5. 노출도: ILO 점수 → KSCO 중분류

세분류 노출도 = 연결된 ISCO 점수의 단순평균, 중분류 노출도 = 소속 세분류의 단순평균. 소분류별 취업자 가중은 자료가 없어 하지 않았다.
대신 (1) 2023년 점수, (2) 대분류를 벗어난 연결을 뺀 계산과 순위를 비교한다.

In [ ]:
ilo = pd.read_excel(RAW / "ilo_wp140" / "Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx").drop_duplicates("ISCO_08")
ilo["isco"] = ilo.ISCO_08.astype(str).str.zfill(4)
lk = pd.read_excel(next((RAW / "ksco_isco").glob("*.xlsx")), sheet_name="4-1. (연계표) 세분류 연계표(KSCO-ISCO)", dtype=str)
lk = lk.rename(columns={"한국코드": "ksco", "국제코드": "isco", "대분류 벗어난 연계": "cross"})
lk["ksco"], lk["isco"], lk["cross"] = lk.ksco.str.strip(), lk.isco.str.strip().str.zfill(4), lk.cross.astype(float) > 0
m = lk.merge(ilo[["isco", "mean_score_2025", "mean_score_2023"]], on="isco", how="left")


def to_mid(df, col):
    unit = df.groupby("ksco")[col].mean()
    return unit.groupby(unit.index.str[:2]).mean()


expo = pd.DataFrame({
    "exp25": to_mid(m, "mean_score_2025"),
    "exp23": to_mid(m, "mean_score_2023"),
    "exp25_nocross": to_mid(m[~m.cross], "mean_score_2025"),
}).drop(index="A0", errors="ignore")
expo["exp_z"] = (expo.exp25 - expo.exp25.mean()) / expo.exp25.std()
expo["terc"] = pd.qcut(expo.exp25, 3, labels=ORDER).astype(str)
assert set(mid.code) == set(expo.index), "KOSIS 중분류와 노출도 코드가 다름"

rank_corr = expo[["exp25", "exp23", "exp25_nocross"]].corr(method="spearman")
display(rank_corr.round(3))

mid = mid.merge(expo, left_on="code", right_index=True)
y_base = mid[mid.year == 2022].groupby("code").y15_29.mean().rename("y_base")
terc_tbl = expo.join(y_base).groupby("terc").agg(직업수=("exp25", "size"), 노출도_평균=("exp25", "mean"),
                                                청년취업자_2022=("y_base", "sum")).loc[ORDER]
terc_tbl["청년_비중"] = terc_tbl.청년취업자_2022 / terc_tbl.청년취업자_2022.sum()
display(terc_tbl.round(3))

In [ ]:
names = mid.drop_duplicates("code").set_index("code").name
d = expo.sort_values("exp25")
fig, ax = plt.subplots(figsize=(7, 10.5))
ax.barh([f"{c} {names[c]}" for c in d.index], d.exp25, color=d.terc.map(TERC), height=0.7)
ax.set(title="KSCO 중분류별 생성형 AI 노출도 (ILO WP140, 2025년 점수)", xlabel="노출도 (0~1)", ylabel="", ylim=(-0.7, len(d) - 0.3))
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=7, length=0)
ax.legend(handles=[Patch(color=TERC[k], label=f"노출도 {k}") for k in reversed(ORDER)], frameon=False, fontsize=8, loc="lower right")
fig.savefig(FIG / "fig4_중분류별_노출도.png")

## E6. 모델 성능 지수 (ECI)

시점별 "업그레이드 수준"은 조사 기준일(4월 15일·10월 15일)까지 출시된 모델의 ECI 최댓값으로 둔다.

In [ ]:
eci = pd.read_csv(RAW / "epoch_eci" / "eci_scores.csv", parse_dates=["date"]).sort_values("date")
eci["frontier"] = eci.eci.cummax()
ref = pd.DataFrame({"period": sorted(p for p in mid.period.unique() if p >= "202301")})
ref["ref_date"] = pd.to_datetime(ref.period.str[:4] + ref.period.str[4:].map({"01": "-04-15", "02": "-10-15"}))
top = [eci[eci.date <= d].nlargest(1, "eci").iloc[0] for d in ref.ref_date]
ref["eci"], ref["model"] = [r.eci for r in top], [r["Display name"] for r in top]
ref["d_eci"] = ref.eci.diff()
r_time = np.corrcoef(np.arange(len(ref)), ref.eci)[0, 1]
display(ref.round(2))
print(f"처치 후 {len(ref)}개 시점에서 ECI 최댓값과 시점 번호의 상관: {r_time:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.scatter(eci.date, eci.eci, s=14, color=AXIS, edgecolor=SURFACE, linewidth=0.5, label="개별 모델", zorder=2)
ax.step(eci.date, eci.frontier, where="post", color=BLUE, lw=2, label="최고 성능(누적 최댓값)", zorder=3)
ax.scatter(ref.ref_date, ref.eci, s=46, color=BLUE, edgecolor=SURFACE, linewidth=2, label="조사 기준일의 값", zorder=4)
for r in ref.itertuples():
    ax.annotate(f"{r.eci:.0f}", (r.ref_date, r.eci), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8, color=INK2)
ax.set(title="Epoch Capabilities Index: 모델별 점수와 최고 성능", xlabel="", ylabel="ECI")
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.savefig(FIG / "fig5_ECI.png")

## E7. 핵심 그림: 노출도 3분위별 취업자 추이

중분류 56개를 노출도 순으로 세 묶음으로 나누고, 묶음별 취업자 합계를 2022년 평균 = 100으로 지수화한다.

In [ ]:
g = mid.groupby(["terc", "period", "time", "year"])[["y15_29", "a30_59", "total"]].sum().reset_index()
b22 = g[g.year == 2022].groupby("terc")[["y15_29", "a30_59"]].mean()
for col in ("y15_29", "a30_59"):
    g[col + "_idx"] = 100 * g[col] / g.terc.map(b22[col])
g["y_share"] = 100 * g.y15_29 / g.groupby("period").y15_29.transform("sum")   # 청년 취업자 중 이 묶음의 비중
g["y_within"] = 100 * g.y15_29 / g.total                                      # 이 묶음 취업자 중 청년 비중


def tercile_lines(axes, cols, titles):
    for ax, col, title in zip(axes, cols, titles):
        for k in ORDER:
            s = g[g.terc == k].sort_values("time")
            ax.plot(s.time, s[col], color=TERC[k], lw=2, label=f"노출도 {k}")
            end_label(ax, s.time.iloc[-1], s[col].iloc[-1], k)
        mark_chatgpt(ax)
        ax.set(title=title, xlabel="", xlim=(2013, 2027.2))


fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), sharey=True)
tercile_lines(axes, ("y15_29_idx", "a30_59_idx"), ("청년(15~29세)", "30~59세"))
for ax in axes:
    ax.axhline(100, color=AXIS, lw=0.8, zorder=1)
axes[0].set_ylabel("취업자 지수 (2022년 평균 = 100)")
axes[0].legend(frameon=False, fontsize=8, loc="lower left")
fig.suptitle("노출도 3분위별 취업자 추이", x=0.01, y=1.02, ha="left", fontsize=11, fontweight="bold")
fig.savefig(FIG / "fig6_노출도3분위_취업자지수.png")

latest = g[g.period == "202601"].set_index("terc").loc[ORDER, ["y15_29_idx", "a30_59_idx", "y_within"]]
latest["y_within_2022"] = g[g.year == 2022].groupby("terc").y_within.mean()
display(latest.rename(columns={"y15_29_idx": "청년 지수(2026 상)", "a30_59_idx": "30~59세 지수(2026 상)", "y_within": "청년 비중 %(2026 상)", "y_within_2022": "청년 비중 %(2022)"}).round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9))
tercile_lines(axes, ("y_share", "y_within"), ("청년 취업자 중 각 묶음의 비중", "각 묶음 취업자 중 청년의 비중"))
axes[0].set_ylabel("%")
axes[0].legend(frameon=False, fontsize=8, loc="center left")
fig.suptitle("인구 감소 영향을 뺀 비중으로 본 추이", x=0.01, y=1.02, ha="left", fontsize=11, fontweight="bold")
fig.savefig(FIG / "fig7_노출도3분위_청년비중.png")

### 격차 추이와 처치 전 추세

격차 = ln(노출도 상 묶음 취업자) - ln(비교 묶음 취업자). 2022년 평균을 0으로 맞췄고 단위는 로그포인트 × 100(대략 %)이다.
"청년 - 30~59세"는 같은 직업 묶음 안에서 청년이 30~59세보다 얼마나 더 줄었는지를 뜻한다(삼중차분의 기술통계).

In [ ]:
w = g.pivot(index="time", columns="terc", values=["y15_29", "a30_59"])


def gap_series(ref_group):
    out = pd.DataFrame({"청년": np.log(w["y15_29"]["상"] / w["y15_29"][ref_group]),
                        "30~59세": np.log(w["a30_59"]["상"] / w["a30_59"][ref_group])}) * 100
    out["청년 - 30~59세"] = out["청년"] - out["30~59세"]
    return out - out[(out.index > 2022) & (out.index < CHATGPT)].mean()


gaps = {"상 - 하": gap_series("하"), "상 - 중": gap_series("중")}
GAPC = {"청년": BLUE, "30~59세": INK2, "청년 - 30~59세": "#eb6834"}
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True)
for ax, (title, gp) in zip(axes, gaps.items()):
    for col in gp.columns:
        ax.plot(gp.index, gp[col], color=GAPC[col], lw=2.4 if "-" in col else 1.8, label=col)
        end_label(ax, gp.index[-1], gp[col].iloc[-1], col)
    ax.axhline(0, color=AXIS, lw=0.8, zorder=1)
    mark_chatgpt(ax)
    ax.set(title=f"노출도 {title}", xlabel="", xlim=(2013, 2029.4))
axes[0].set_ylabel("격차 (2022년 평균 = 0)")
axes[1].legend(frameon=False, fontsize=8, loc="lower left")
fig.suptitle("노출도 높은 묶음과 비교 묶음의 취업자 격차", x=0.01, y=1.02, ha="left", fontsize=11, fontweight="bold")
fig.savefig(FIG / "fig8_격차_추이.png")

levels = pd.concat({k: pd.concat([gp.groupby(gp.index.astype(int)).mean().loc[[2013, 2016, 2019, 2022, 2024]],
                                  gp[gp.index > 2025.5].mean().to_frame("최근(25 하·26 상)").T]) for k, gp in gaps.items()})
display(levels.round(1))

post_t = gaps["상 - 하"].index[gaps["상 - 하"].index > CHATGPT]
rows = []
for cname, gp in gaps.items():
    for label, pre_t in {"2013~2022": gp.index[gp.index < CHATGPT], "2013~2019": gp.index[gp.index < 2020]}.items():
        for col in gp.columns:
            slope, icpt = np.polyfit(pre_t, gp.loc[pre_t, col], 1)
            dev = gp.loc[post_t, col] - (icpt + slope * post_t)
            rows.append({"비교": cname, "추세 기간": label, "격차": col, "처치 전 기울기(연간)": slope,
                         "처치 후 이탈 평균": dev.mean(), "최근 2개 시점 이탈": dev.iloc[-2:].mean()})
trend = pd.DataFrame(rows).set_index(["비교", "추세 기간", "격차"])
display(trend.round(2))

## E8. 직업별 산점도: 노출도와 취업자 변화

계절성을 맞추기 위해 상·하반기 평균끼리 비교한다. 세 구간 모두 길이는 3.5년이다. 기준 취업자가 1천명 미만인 직업은 로그 변화를 계산할 수 없어 제외한다.

In [ ]:
WINDOWS = {
    "처치 후 (2022 → 2025 하·2026 상)": (["202201", "202202"], ["202502", "202601"]),
    "플라시보 1 (2015 → 2018 하·2019 상)": (["201501", "201502"], ["201802", "201901"]),
    "플라시보 2 (2018 하·2019 상 → 2022)": (["201802", "201901"], ["202201", "202202"]),
}


def window_change(col, pre, post):
    a = mid[mid.period.isin(pre)].groupby("code")[col].mean()
    b = mid[mid.period.isin(post)].groupby("code")[col].mean()
    with np.errstate(divide="ignore"):
        out = pd.DataFrame({"base": a, "dln": 100 * np.log(b / a)}).join(expo[["exp25", "exp_z"]])
    return out[(out.base >= 1) & np.isfinite(out.dln)]


def changes(pre, post):
    """청년, 30~59세, 그리고 둘의 차이(같은 직업 안에서 청년이 더 줄어든 정도). 가중치는 청년 기준 취업자."""
    y, a = window_change("y15_29", pre, post), window_change("a30_59", pre, post)
    diff = y.join(a.dln.rename("dln_a"), how="inner")
    diff["dln"] = diff.dln - diff.dln_a
    return {"청년": y, "30~59세": a, "청년 - 30~59세": diff}


def wslope(df, x="exp_z"):
    return np.polyfit(df[x], df.dln, 1, w=np.sqrt(df.base))[0]


rows = []
for wname, (pre, post) in WINDOWS.items():
    for lab, dfw in changes(pre, post).items():
        rows.append({"구간": wname, "대상": lab, "직업 수": len(dfw), "가중 기울기": wslope(dfw), "단순 기울기": np.polyfit(dfw.exp_z, dfw.dln, 1)[0]})
slopes = pd.DataFrame(rows).set_index(["구간", "대상"])
print("노출도 1표준편차당 취업자 변화 (로그포인트 × 100)")
display(slopes.round(2))

MAIN = "처치 후 (2022 → 2025 하·2026 상)"
sc = changes(*WINDOWS[MAIN])
loo = {lab: pd.Series({c: wslope(sc[lab].drop(index=c)) for c in sc[lab].index}) for lab in ("청년", "청년 - 30~59세")}
for lab, s_ in loo.items():
    print(f"{lab}: 직업 하나씩 빼고 다시 계산한 가중 기울기 {s_.min():.2f} ~ {s_.max():.2f} "
          f"(가장 크게 바꾸는 직업: {s_.idxmin()} {names[s_.idxmin()]}, {s_.idxmax()} {names[s_.idxmax()]})")

YLIM = (-90, 45)
to_label = sc["청년"].base.nlargest(8).index
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3), sharey=True)
for ax, (lab, dfw) in zip(axes, sc.items()):
    ax.scatter(dfw.exp25, dfw.dln, s=6 + dfw.base / dfw.base.max() * 420, color=BLUE, alpha=0.75, edgecolor=SURFACE, linewidth=1.2, zorder=3)
    s1, s0 = np.polyfit(dfw.exp25, dfw.dln, 1, w=np.sqrt(dfw.base))
    xs = np.array([dfw.exp25.min(), dfw.exp25.max()])
    ax.plot(xs, s0 + s1 * xs, color=INK2, lw=1.5, zorder=4)
    ax.axhline(0, color=AXIS, lw=0.8, zorder=1)
    for c in to_label:
        if c in dfw.index:
            ax.annotate(c, (dfw.exp25[c], dfw.dln[c]), ha="center", va="center", fontsize=7, color="white", fontweight="bold", zorder=5)
    n_out = int(((dfw.dln < YLIM[0]) | (dfw.dln > YLIM[1])).sum())
    if n_out:
        ax.text(0.02, 0.03, f"축 범위 밖 소규모 직업 {n_out}개", transform=ax.transAxes, fontsize=7.5, color=MUTED)
    ax.set(title=lab, xlabel="노출도 (0~1)", ylim=YLIM)
axes[0].set_ylabel("변화, 2022 → 2025 하·2026 상 (로그포인트 × 100)")
fig.suptitle("노출도와 취업자 변화 — 원 크기는 2022년 취업자 수, 선은 가중 추세선, 숫자는 직업 코드", x=0.01, y=1.02, ha="left", fontsize=11, fontweight="bold")
fig.savefig(FIG / "fig9_노출도_변화_산점도.png")

lookup = sc["청년"].loc[to_label, ["exp25", "base", "dln"]].join(sc["30~59세"].dln.rename("dln_30_59"))
lookup.insert(0, "직업", names[lookup.index])
display(lookup.rename(columns={"exp25": "노출도", "base": "청년 취업자(2022, 천명)", "dln": "청년 변화", "dln_30_59": "30~59세 변화"}).round(2))

## E9. 진입 판단

In [ ]:
s = slopes["가중 기울기"]
fmt3 = lambda lab: " / ".join(f"{s[(k, lab)]:.2f}" for k in WINDOWS)   # 처치 후 / 플라시보 1 / 플라시보 2
pre_slope = lambda lab: trend.loc[("상 - 하", "2013~2022", lab), "처치 전 기울기(연간)"]
same_sign = lambda x: bool(np.sign(x.min()) == np.sign(x.max()))
checks = pd.DataFrame([
    ("E1 합계 일치", f"최대 차이 {max(d_mid, d_maj, d_age):.0f}천명", max(d_mid, d_maj, d_age) <= 3),
    ("E4 신뢰 가능한 셀의 청년 취업자 비중 ≥ 90%", f"{1 - share_emp:.1%}", 1 - share_emp >= 0.9),
    ("E5 노출도가 붙은 중분류", f"{expo.exp25.notna().sum()}/{mid.code.nunique()}", bool(expo.exp25.notna().all())),
    ("E5 계산 방식 간 순위상관 ≥ 0.9", f"최소 {rank_corr.values.min():.3f}", bool(rank_corr.values.min() >= 0.9)),
    ("E7 처치 전 기울기(연간, 상-하): 청년", f"{pre_slope('청년'):.2f}", "판단 필요"),
    ("E7 처치 전 기울기(연간, 상-하): 청년 - 30~59세", f"{pre_slope('청년 - 30~59세'):.2f}", "판단 필요"),
    ("E8 가중 기울기, 청년 (처치 후 / 플라시보 1 / 2)", fmt3("청년"), "판단 필요"),
    ("E8 가중 기울기, 청년 - 30~59세 (처치 후 / 플라시보 1 / 2)", fmt3("청년 - 30~59세"), "판단 필요"),
    ("E8 직업 하나를 빼도 부호 유지: 청년", f"{loo['청년'].min():.2f} ~ {loo['청년'].max():.2f}", same_sign(loo["청년"])),
    ("E8 직업 하나를 빼도 부호 유지: 청년 - 30~59세", f"{loo['청년 - 30~59세'].min():.2f} ~ {loo['청년 - 30~59세'].max():.2f}", same_sign(loo["청년 - 30~59세"])),
], columns=["점검", "값", "통과"])
display(checks)

cols = ["code", "name", "major", "period", "year", "half", "time", "post", "total", "y15_29", "a30_39", "a40_49", "a50_59", "a60p",
        "a30_59", "y_flag", "exp25", "exp23", "exp25_nocross", "exp_z", "terc"]
mid[cols].to_csv(OUT / "panel_mid.csv", index=False, encoding="utf-8-sig")
ref.to_csv(OUT / "eci_ref.csv", index=False, encoding="utf-8-sig")
print("저장:", OUT / "panel_mid.csv", mid[cols].shape, "|", OUT / "eci_ref.csv", ref.shape)

## 요약과 판단 (2026-11-11 실행 기준)

**데이터 품질은 모두 통과했다.** 합계 차이 최대 3천명, 신뢰 가능한 셀의 청년 취업자 비중 99.6%, 노출도 매칭 56/56, 계산 방식 간 순위상관 최소 0.942.

**보이는 것**

1. 2022년 이후 청년 취업자는 노출도 세 묶음 모두에서 줄었다(2022년 평균 = 100 → 2026 상: 상 82.1, 중 90.7, 하 84.6).
2. 노출도 상 묶음에서는 30~59세가 늘었는데(108.6) 청년만 줄었다. 이 묶음의 청년 비중은 17.3% → 13.3%로, 중(16.3 → 14.4)·하(7.3 → 6.4)보다 크게 떨어졌다.

**그러나 "2022년 이후에 새로 생긴 변화"라는 근거는 약하다**

3. 청년만 보면 노출도 상-하 격차는 2013~2022년에 이미 연 2.9씩 줄었고, 2022년 이후에는 거의 변하지 않았다(+0.6).
4. 청년 - 30~59세 격차(상-하)는 2022년 이전부터 연 4.5씩 줄어 왔고 이후에도 같은 속도다(추세 연장선과의 차이 0.4~2.5).
5. 직업 단위 기울기도 같은 결론이다. 노출도 1표준편차당 청년 - 30~59세 변화는 처치 후 -7.6, 처치 전 두 구간 -6.5와 -3.5였다. 청년만의 기울기(-2.0)는 처치 전(-2.5, -2.8)보다 작고, 직업 하나(45 조리 및 음식 서비스직)를 빼면 부호가 바뀐다.
6. 상-중 비교에서만 2022년 이후 꺾임이 보인다. 청년 - 30~59세 격차가 처치 전 연 -0.6에서 이후 3.5년간 -15.3이 됐다.

**판단**

- 청년만의 단순 이중차분(M2)은 평행추세 가정이 성립하지 않는다. 설계 3-5의 기준대로 인과 해석을 하지 않고 "기존 추세의 연장"으로 보고한다.
- 회귀 단계의 주 분석은 직업 안의 청년 대 30~59세 비교(M3)로 옮기고, 수준 변화가 아니라 2022년 이후 추세가 꺾였는지를 검정한다.
- 하 묶음은 청년 취업자의 14.6%뿐이고 2013~2020년에 청년이 크게 늘어난 뒤 줄어드는 등 움직임이 다르다. 상-하와 상-중의 결과가 갈리는 이유를 직업별로 확인해야 한다.